In [10]:
import pickle
import torch
from torch import nn
from nltk.tokenize import word_tokenize
import re

In [11]:
# ── 1. Load vocab and config ──────────────────────────────────────────────────
with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

with open('config.pkl', 'rb') as f:
    config = pickle.load(f)

max_len = config['max_len']

In [12]:
# ── 2. Copy the exact same model class ───────────────────────────────────────
class LSTMmodel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=0.3)
        self.dropout    = nn.Dropout(0.4)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.fc         = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        pooled = lstm_out.mean(dim=1)
        pooled = self.dropout(pooled)
        pooled = self.layer_norm(pooled)
        return self.fc(pooled)

In [13]:
# ── 3. Load the saved weights ─────────────────────────────────────────────────
model = LSTMmodel(
    vocab_size = config['vocab_size'],
    embed_dim  = 100,
    hidden_dim = 256,
    output_dim = 6
)
model.load_state_dict(torch.load('best_model.pt', map_location='cpu'))
model.eval()
print("Model loaded successfully")

Model loaded successfully


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19636\611663326.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pt', map_locati

In [14]:
# ── 4. Copy the same preprocessing functions ──────────────────────────────────
from nltk.corpus import stopwords
KEEP_NEGATIONS = {"not","no","nor","never","don't","didn't","won't","can't","couldn't","shouldn't","wouldn't"}
stop_words = set(stopwords.words('english')) - KEEP_NEGATIONS

def clear_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in stop_words]

def encode(tokens):
    return [vocab.get(w, 1) for w in tokens]

def pad(seq):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))

In [15]:
# ── 5. Predict on a single sentence ──────────────────────────────────────────
label_map = {0:"sadness", 1:"joy", 2:"love", 3:"anger", 4:"fear", 5:"surprise"}

def predict(text):
    tokens  = clear_text(text)
    encoded = encode(tokens)
    padded  = pad(encoded)
    tensor  = torch.tensor([padded], dtype=torch.long)   # shape [1, max_len]

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]
        pred   = probs.argmax().item()

    return {
        'emotion'     : label_map[pred],
        'confidence'  : f"{probs[pred]*100:.1f}%",
        'all_scores'  : {label_map[i]: f"{p*100:.1f}%" for i, p in enumerate(probs)}
    }

In [16]:
# ── 6. Test it ────────────────────────────────────────────────────────────────
print(predict("I am so happy today!"))
print(predict("I can't believe they did that to me"))

{'emotion': 'joy', 'confidence': '97.8%', 'all_scores': {'sadness': '0.1%', 'joy': '97.8%', 'love': '1.4%', 'anger': '0.4%', 'fear': '0.0%', 'surprise': '0.3%'}}
{'emotion': 'surprise', 'confidence': '99.3%', 'all_scores': {'sadness': '0.1%', 'joy': '0.2%', 'love': '0.0%', 'anger': '0.4%', 'fear': '0.1%', 'surprise': '99.3%'}}
